# Experiment 2 — Create a Blockchain using Python

Aim: Create a Blockchain in Python with block creation, Proof-of-Work mining, chain validation, and pending-transaction handling, then demonstrate it over a Flask HTTP API.

## Step 1: Blockchain class

Core logic: block creation, Proof-of-Work with a `"000"`-leading-zero puzzle (as required by the task, tightened from the sample `"0000"`), pending-transaction handling (`create_Transactions`, block only mined when transactions exist, transactions cleared after mining), and chain validation.

In [1]:
import datetime
import hashlib
import json as jsonlib

class Blockchain:
    def __init__(self):
        self.chain = []
        self.transactions = []
        self.create_block(proof=1, previous_hash='0')

    def create_block(self, proof, previous_hash):
        block = {
            'index': len(self.chain) + 1,
            'timestamp': str(datetime.datetime.now()),
            'proof': proof,
            'previous_hash': previous_hash,
            'transactions': self.transactions,
        }
        self.transactions = []  # clear pending transactions after mining
        self.chain.append(block)
        return block

    def create_Transactions(self, sender, receiver, amount):
        self.transactions.append({'sender': sender, 'receiver': receiver, 'amount': amount})
        return self.get_previous_block()['index'] + 1

    def get_previous_block(self):
        return self.chain[-1]

    def proof_of_work(self, previous_proof):
        new_proof = 1
        check_proof = False
        while not check_proof:
            hash_operation = hashlib.sha256(
                str(new_proof**2 - previous_proof**2).encode()
            ).hexdigest()
            if hash_operation[:3] == '000':   # golden-nonce puzzle: 3 leading zeros
                check_proof = True
            else:
                new_proof += 1
        return new_proof

    def hash(self, block):
        encoded_block = jsonlib.dumps(block, sort_keys=True).encode()
        return hashlib.sha256(encoded_block).hexdigest()

    def is_chain_valid(self, chain):
        previous_block = chain[0]
        block_index = 1
        while block_index < len(chain):
            block = chain[block_index]
            if block['previous_hash'] != self.hash(previous_block):
                return False
            previous_proof = previous_block['proof']
            proof = block['proof']
            hash_operation = hashlib.sha256(
                str(proof**2 - previous_proof**2).encode()
            ).hexdigest()
            if hash_operation[:3] != '000':
                return False
            previous_block = block
            block_index += 1
        return True

blockchain = Blockchain()
print("Genesis block created:", blockchain.chain[0])


Genesis block created: {'index': 1, 'timestamp': '2026-09-09 11:49:36.507440', 'proof': 1, 'previous_hash': '0', 'transactions': []}


## Step 2: Mine only when transactions are pending

Mining is gated on `self.transactions` being non-empty, and the pending list is cleared as part of `create_block` once mined.

In [2]:
def mine_block(bc):
    if not bc.transactions:
        return {"message": "No pending transactions — nothing to mine."}
    previous_block = bc.get_previous_block()
    previous_proof = previous_block['proof']
    proof = bc.proof_of_work(previous_proof)
    previous_hash = bc.hash(previous_block)
    block = bc.create_block(proof, previous_hash)
    return {
        'message': 'Congratulations, you just mined a block!',
        'index': block['index'],
        'timestamp': block['timestamp'],
        'proof': block['proof'],
        'previous_hash': block['previous_hash'],
        'transactions': block['transactions'],
    }

# Attempt to mine with no pending transactions
print("Mine with empty pool:", mine_block(blockchain))

# Add transactions, then mine
blockchain.create_Transactions("Alice", "Bob", 200)
blockchain.create_Transactions("Bob", "Dave", 500)
print("Pending before mining:", blockchain.transactions)
print("Mine result:", mine_block(blockchain))
print("Pending after mining :", blockchain.transactions)


Mine with empty pool: {'message': 'No pending transactions — nothing to mine.'}
Pending before mining: [{'sender': 'Alice', 'receiver': 'Bob', 'amount': 200}, {'sender': 'Bob', 'receiver': 'Dave', 'amount': 500}]
Mine result: {'message': 'Congratulations, you just mined a block!', 'index': 2, 'timestamp': '2026-09-09 11:49:36.513938', 'proof': 533, 'previous_hash': '43e7548729265b70730c0b4de2eaa19dc5716167ce2d39f23c25341fe5ee770f', 'transactions': [{'sender': 'Alice', 'receiver': 'Bob', 'amount': 200}, {'sender': 'Bob', 'receiver': 'Dave', 'amount': 500}]}
Pending after mining : []


## Step 3: Inspect the chain and validate it

In [3]:
for block in blockchain.chain:
    print(block)

print("\nIs chain valid?", blockchain.is_chain_valid(blockchain.chain))


{'index': 1, 'timestamp': '2026-09-09 11:49:36.507440', 'proof': 1, 'previous_hash': '0', 'transactions': []}
{'index': 2, 'timestamp': '2026-09-09 11:49:36.513938', 'proof': 533, 'previous_hash': '43e7548729265b70730c0b4de2eaa19dc5716167ce2d39f23c25341fe5ee770f', 'transactions': [{'sender': 'Alice', 'receiver': 'Bob', 'amount': 200}, {'sender': 'Bob', 'receiver': 'Dave', 'amount': 500}]}

Is chain valid? True


## Step 4: Expose the Blockchain over a Flask API

The same class is wrapped in a Flask app with `/mine_block`, `/get_chain`, and `/is_valid` routes, run in a background thread inside this notebook so it can be exercised with real HTTP requests (in place of a separate Postman session).

In [4]:
from flask import Flask, jsonify, request
import threading

app = Flask(__name__)
api_blockchain = Blockchain()

@app.route('/mine_block', methods=['GET'])
def route_mine_block():
    return jsonify(mine_block(api_blockchain)), 200

@app.route('/add_transaction', methods=['POST'])
def route_add_transaction():
    data = request.get_json()
    index = api_blockchain.create_Transactions(data['sender'], data['receiver'], data['amount'])
    return jsonify({'message': f'Transaction will be added to Block {index}'}), 201

@app.route('/get_chain', methods=['GET'])
def route_get_chain():
    return jsonify({'chain': api_blockchain.chain, 'length': len(api_blockchain.chain)}), 200

@app.route('/is_valid', methods=['GET'])
def route_is_valid():
    valid = api_blockchain.is_chain_valid(api_blockchain.chain)
    message = 'All good. The Blockchain is valid.' if valid else 'Houston, we have a problem. The Blockchain is not valid.'
    return jsonify({'message': message}), 200

server_thread = threading.Thread(
    target=lambda: app.run(host='127.0.0.1', port=5050, use_reloader=False),
    daemon=True,
)
server_thread.start()
import time; time.sleep(1)
print("Flask API running on http://127.0.0.1:5050")


 * Serving Flask app '__main__'


 * Debug mode: off


 * Running on http://127.0.0.1:5050


Press CTRL+C to quit


Flask API running on http://127.0.0.1:5050


## Step 5: Exercise the API with `requests` (in place of Postman)

In [5]:
import requests

base = "http://127.0.0.1:5050"

r = requests.get(f"{base}/mine_block")
print("GET /mine_block (no transactions yet):", r.json())

r = requests.post(f"{base}/add_transaction", json={"sender": "Alice", "receiver": "Bob", "amount": 200})
print("POST /add_transaction:", r.json())

r = requests.post(f"{base}/add_transaction", json={"sender": "Bob", "receiver": "Dave", "amount": 500})
print("POST /add_transaction:", r.json())

r = requests.get(f"{base}/mine_block")
print("GET /mine_block (with transactions):", r.json())

r = requests.get(f"{base}/get_chain")
print("GET /get_chain:", r.json())

r = requests.get(f"{base}/is_valid")
print("GET /is_valid:", r.json())


127.0.0.1 - - [09/Sep/2026 11:49:37] "GET /mine_block HTTP/1.1" 200 -


127.0.0.1 - - [09/Sep/2026 11:49:38] "POST /add_transaction HTTP/1.1" 201 -


127.0.0.1 - - [09/Sep/2026 11:49:38] "POST /add_transaction HTTP/1.1" 201 -


127.0.0.1 - - [09/Sep/2026 11:49:38] "GET /mine_block HTTP/1.1" 200 -


127.0.0.1 - - [09/Sep/2026 11:49:38] "GET /get_chain HTTP/1.1" 200 -


127.0.0.1 - - [09/Sep/2026 11:49:38] "GET /is_valid HTTP/1.1" 200 -


GET /mine_block (no transactions yet): {'message': 'No pending transactions — nothing to mine.'}
POST /add_transaction: {'message': 'Transaction will be added to Block 2'}
POST /add_transaction: {'message': 'Transaction will be added to Block 2'}
GET /mine_block (with transactions): {'index': 2, 'message': 'Congratulations, you just mined a block!', 'previous_hash': '951ab7233d17ca22234dcfd86fc011dd45c537d2e65994bc952a68a4bfc301d9', 'proof': 533, 'timestamp': '2026-09-09 11:49:38.102074', 'transactions': [{'amount': 200, 'receiver': 'Bob', 'sender': 'Alice'}, {'amount': 500, 'receiver': 'Dave', 'sender': 'Bob'}]}
GET /get_chain: {'chain': [{'index': 1, 'previous_hash': '0', 'proof': 1, 'timestamp': '2026-09-09 11:49:36.632923', 'transactions': []}, {'index': 2, 'previous_hash': '951ab7233d17ca22234dcfd86fc011dd45c537d2e65994bc952a68a4bfc301d9', 'proof': 533, 'timestamp': '2026-09-09 11:49:38.102074', 'transactions': [{'amount': 200, 'receiver': 'Bob', 'sender': 'Alice'}, {'amount': 500

## Observations

- Tightening the Proof-of-Work target from `"0000"` to `"000"` (3 leading zeros) reduces the average number of nonce trials needed to mine a block, since a shorter required prefix is satisfied more often by chance.
- Gating `mine_block` on a non-empty transaction pool avoids wasting Proof-of-Work effort on empty blocks, and clearing the pool inside `create_block` prevents the same transactions from being mined twice.
- Driving the Flask API with the `requests` library from inside the notebook reproduces exactly what Postman would do (JSON POST/GET over HTTP) while keeping the whole demonstration reproducible in one execution.